## 1. Install Required Libraries

In [2]:
!pip install pandas networkx shapely pyproj scipy datasets matplotlib

## 2. Build Road Network & Dijkstra (Takes ~2 mins)

In [ ]:
import pandas as pd
import networkx as nx
from shapely import wkt
from shapely.geometry import Point, LineString
import math
import json
from pyproj import Transformer
from scipy.spatial import cKDTree
from datasets import load_dataset
import datetime

print("1. Loading VRP delivery nodes...")

TARGET_DATE = 708
TARGET_COURIER = 1043

try:
    NUM_NODES = 15
except (IndexError, ValueError):
    NUM_NODES = 15

print(f"Building matrix graph for {NUM_NODES} customers...")

dataset = load_dataset("Cainiao-AI/LaDe-D", split="delivery_sh")
df_nodes = dataset.to_pandas()
df_subset = df_nodes[(df_nodes['ds'] == TARGET_DATE) & (df_nodes['courier_id'] == TARGET_COURIER)].head(NUM_NODES).reset_index()

nodes_wgs84 = {'Depot': (121.52000, 31.08000)}
for i, row in df_subset.iterrows():
    nodes_wgs84[f"N{i+1}"] = (float(row['lng']), float(row['lat']))

# Convert to EPSG:3857
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
nodes_epsg3857 = {}
for name, coords in nodes_wgs84.items():
    x, y = transformer.transform(coords[0], coords[1])
    nodes_epsg3857[name] = (x, y)
    print(f"{name} -> EPSG:3857 ({x:.2f}, {y:.2f})")

print("\n2. Reading roads_shanghai.csv and assigning speeds (Class-Based Imputation)...")
df_roads = pd.read_csv('/content/roads_shanghai.csv', sep='\t')
df_roads['maxspeed'] = pd.to_numeric(df_roads['maxspeed'], errors='coerce').fillna(0)

class_speeds = {
    'motorway': 95.0, 'trunk': 70.6, 'primary': 59.0, 'tertiary_link': 53.3,
    'secondary': 51.4, 'primary_link': 47.3, 'trunk_link': 45.8, 'motorway_link': 41.5,
    'tertiary': 40.8, 'secondary_link': 37.5, 'cycleway': 35.0, 'residential': 24.7,
    'unclassified': 23.6, 'service': 18.4, 'living_street': 17.0, 'path': 15.0, 'pedestrian': 9.0,
    # Unpaved tracks & non-motorised paths (previously silently used the 30 km/h default)
    'track': 15.0, 'track_grade1': 20.0, 'track_grade2': 15.0, 'track_grade3': 12.0,
    'track_grade5': 5.0, 'footway': 5.0, 'steps': 2.0, 'bridleway': 5.0
}
default_speed = 30.0 # fallback for any unrecognised / unknown class

def get_speed(row):
    if row['maxspeed'] > 0:
        return float(row['maxspeed'])
    return class_speeds.get(row['fclass'], default_speed)

df_roads['assigned_speed'] = df_roads.apply(get_speed, axis=1)

print("3. Building the road network graph (NetworkX)...")
G = nx.DiGraph()

for idx, row in df_roads.iterrows():
    try:
        geom = wkt.loads(row['geometry'])
    except:
        continue
        
    if not isinstance(geom, LineString):
        continue
        
    coords = list(geom.coords)
    oneway = row['oneway']
    speed_kmh = row['assigned_speed']
    fclass = row['fclass']

    for i in range(len(coords) - 1):
        u = coords[i]
        v = coords[i+1]

        # Distance in meters (EPSG:3857 is in meters)
        dist_m = math.sqrt((u[0] - v[0])**2 + (u[1] - v[1])**2)
        if dist_m == 0: continue

        time_hours = (dist_m / 1000.0) / speed_kmh
        dist_km = dist_m / 1000.0

        # Add edges according to oneway direction (store fclass & speed for the per-leg breakdown)
        if oneway == 'F':
            G.add_edge(u, v, weight=time_hours, length=dist_km, fclass=fclass, speed=speed_kmh)
        elif oneway == 'T':
            G.add_edge(v, u, weight=time_hours, length=dist_km, fclass=fclass, speed=speed_kmh)
        else: # 'B' or others
            G.add_edge(u, v, weight=time_hours, length=dist_km, fclass=fclass, speed=speed_kmh)
            G.add_edge(v, u, weight=time_hours, length=dist_km, fclass=fclass, speed=speed_kmh)

print(f"Graph built with {G.number_of_nodes()} junction nodes and {G.number_of_edges()} road edges.")

print("\n4. Finding nearest junctions (Nearest Node Mapping) using KDTree...")
graph_nodes = list(G.nodes())
kdtree = cKDTree(graph_nodes)

node_mapping = {}
for name, target_coords in nodes_epsg3857.items():
    dist, idx = kdtree.query(target_coords)
    mapped_node = graph_nodes[idx]
    node_mapping[name] = mapped_node
    print(f"{name} mapped to nearest junction at {dist:.2f} meters.")

print("\n5. Running Dijkstra for the 16x16 matrix...")
# Compute the shortest path from each mapped node to every other mapped node
node_names = list(nodes_wgs84.keys())

time_matrix = {}
dist_matrix = {}
speed_breakdown = {}  # u -> v -> [ {fclass, dist_km, time_h, speed_kmh}, ... ]
path_geometry = {}    # u -> v -> {coords: [[x,y],...] EPSG:3857, fclass: [per-segment]}

for u_name in node_names:
    time_matrix[u_name] = {}
    dist_matrix[u_name] = {}
    speed_breakdown[u_name] = {}
    path_geometry[u_name] = {}
    u_graph_node = node_mapping[u_name]

    # Run Dijkstra from u to all nodes
    # Since the graph may be disconnected (disconnected components),
    # we must handle the NetworkXNoPath error
    for v_name in node_names:
        if u_name == v_name:
            time_matrix[u_name][v_name] = 0.0
            dist_matrix[u_name][v_name] = 0.0
            speed_breakdown[u_name][v_name] = []
            path_geometry[u_name][v_name] = {'coords': [], 'fclass': []}
            continue

        v_graph_node = node_mapping[v_name]
        try:
            # Shortest path by 'weight' (travel time)
            path = nx.shortest_path(G, source=u_graph_node, target=v_graph_node, weight='weight')

            total_time = 0.0
            total_dist = 0.0
            leg_classes = {}  # fclass -> [dist_km, time_h]
            for k in range(len(path)-1):
                edge_data = G[path[k]][path[k+1]]
                total_time += edge_data['weight']
                total_dist += edge_data['length']
                agg = leg_classes.setdefault(edge_data['fclass'], [0.0, 0.0])
                agg[0] += edge_data['length']
                agg[1] += edge_data['weight']

            # Breakdown by fclass (sorted by descending distance); effective speed = distance/time
            breakdown = [
                {'fclass': fc, 'dist_km': d, 'time_h': t,
                 'speed_kmh': (d / t) if t > 0 else 0.0}
                for fc, (d, t) in sorted(leg_classes.items(), key=lambda x: -x[1][0])
            ]

            time_matrix[u_name][v_name] = total_time
            dist_matrix[u_name][v_name] = total_dist
            speed_breakdown[u_name][v_name] = breakdown
            # Save the actual traversed path geometry (EPSG:3857) + per-segment fclass
            path_geometry[u_name][v_name] = {
                'coords': [[round(px, 1), round(py, 1)] for (px, py) in path],
                'fclass': [G[path[k]][path[k+1]]['fclass'] for k in range(len(path) - 1)]
            }
        except nx.NetworkXNoPath:
            # If there is no connected path (disconnected components),
            # fall back to straight-line (air) distance
            print(f"WARNING: No path found from {u_name} to {v_name}! Falling back to straight-line distance.")
            euclidean_km = math.sqrt((nodes_wgs84[u_name][0] - nodes_wgs84[v_name][0])**2 + (nodes_wgs84[u_name][1] - nodes_wgs84[v_name][1])**2) * 100
            time_matrix[u_name][v_name] = euclidean_km / 20.0 # fallback speed 20
            dist_matrix[u_name][v_name] = euclidean_km
            speed_breakdown[u_name][v_name] = [
                {'fclass': '(air-distance fallback)', 'dist_km': euclidean_km,
                 'time_h': euclidean_km / 20.0, 'speed_kmh': 20.0}
            ]
            path_geometry[u_name][v_name] = {
                'coords': [list(node_mapping[u_name]), list(node_mapping[v_name])],
                'fclass': ['(air-distance fallback)']
            }

print("\n6. Saving the resulting matrices to matrix.json...")
output_data = {
    'nodes_epsg3857': nodes_epsg3857,
    'time_matrix': time_matrix,
    'distance_matrix': dist_matrix,
    'speed_breakdown': speed_breakdown,
    'path_geometry': path_geometry
}

with open('matrix.json', 'w') as f:
    json.dump(output_data, f, indent=4)

print("Done! matrix.json created successfully.")


1. Loading VRP delivery nodes...
Building matrix graph for 15 customers...
Depot -> EPSG:3857 (13527544.52, 3643143.03)
N1 -> EPSG:3857 (13525583.07, 3643127.43)
N2 -> EPSG:3857 (13525580.85, 3643123.53)
N3 -> EPSG:3857 (13525586.41, 3643110.54)
N4 -> EPSG:3857 (13525585.30, 3643117.03)
N5 -> EPSG:3857 (13525704.41, 3643221.02)
N6 -> EPSG:3857 (13525565.26, 3643083.24)
N7 -> EPSG:3857 (13525500.70, 3643465.38)
N8 -> EPSG:3857 (13525633.17, 3643430.29)
N9 -> EPSG:3857 (13525492.90, 3643465.38)
N10 -> EPSG:3857 (13525620.92, 3643547.27)
N11 -> EPSG:3857 (13525449.49, 3643369.19)
N12 -> EPSG:3857 (13525488.45, 3643461.48)
N13 -> EPSG:3857 (13525527.41, 3643477.08)
N14 -> EPSG:3857 (13525520.73, 3643393.89)
N15 -> EPSG:3857 (13525972.69, 3643273.01)

2. Reading roads_shanghai.csv and assigning speeds (Class-Based Imputation)...


FileNotFoundError: [Errno 2] No such file or directory: 'roads_shanghai.csv'

## 3. Run Routing Heuristics (Nearest Neighbor + 2-Opt)

In [ ]:
import pandas as pd
from datasets import load_dataset
import datetime
import math

import json
with open('matrix.json', 'r') as f:
    matrix_data = json.load(f)
time_matrix = matrix_data['time_matrix']
dist_matrix = matrix_data['distance_matrix']
speed_breakdown = matrix_data.get('speed_breakdown', {})
import json
import os

# ========================================================
# 🚀 PROJECT CONFIGURATION SETTINGS
# ========================================================
TARGET_DATE = 708          # Choose the date (e.g., 603, 604, 1027)
TARGET_COURIER = 1043      # Choose the Courier ID
NUM_NODES = len(time_matrix) - 1  # AUTO-SYNC WITH MATRIX.JSON
MAX_DIST_KM = 15.0         # Realistic operational radius (total route distance cap, km)
# ========================================================

# Class-Based Speed Imputation table — MUST mirror build_road_network.py
CLASS_SPEEDS = {
    'motorway': 95.0, 'trunk': 70.6, 'primary': 59.0, 'tertiary_link': 53.3,
    'secondary': 51.4, 'primary_link': 47.3, 'trunk_link': 45.8, 'motorway_link': 41.5,
    'tertiary': 40.8, 'secondary_link': 37.5, 'cycleway': 35.0, 'residential': 24.7,
    'unclassified': 23.6, 'service': 18.4, 'living_street': 17.0, 'path': 15.0, 'pedestrian': 9.0,
    'track': 15.0, 'track_grade1': 20.0, 'track_grade2': 15.0, 'track_grade3': 12.0,
    'track_grade5': 5.0, 'footway': 5.0, 'steps': 2.0, 'bridleway': 5.0
}
DEFAULT_SPEED = 30.0

# Compute how much of the network relies on maxspeed vs the fclass fallback
road_total = road_has_ms = road_fallback = None
try:
    _rd = pd.read_csv('roads_shanghai.csv', sep='\t', usecols=['maxspeed'])
    _ms = pd.to_numeric(_rd['maxspeed'], errors='coerce').fillna(0)
    road_total = len(_ms)
    road_has_ms = int((_ms > 0).sum())
    road_fallback = road_total - road_has_ms
except Exception as e:
    print(f"(Note: could not compute maxspeed coverage: {e})")

print("Downloading dataset...")
dataset = load_dataset("Cainiao-AI/LaDe-D", split="delivery_sh")
df = dataset.to_pandas()

# Filter for the configured courier and date
df_subset = df[(df['ds'] == TARGET_DATE) & (df['courier_id'] == TARGET_COURIER)].head(NUM_NODES).reset_index()

def time_to_hours(time_str):
    t = datetime.datetime.strptime(time_str, "%m-%d %H:%M:%S")
    return t.hour + t.minute / 60.0 + t.second / 3600.0

nodes = {'Depot': (121.52000, 31.08000)}
time_windows = {}

for i, row in df_subset.iterrows():
    n = f"N{i+1}"
    nodes[n] = (float(row['lng']), float(row['lat']))
    e_j = time_to_hours(row['accept_time'])
    l_j = time_to_hours(row['delivery_time'])
    time_windows[n] = (e_j, l_j)

def evaluate_route_detailed(route):
    current_time = 8.00
    total_distance = 0.0
    total_lateness_penalty = 0.0
    total_wait_penalty = 0.0
    
    steps_log = []
    
    for i in range(len(route) - 1):
        curr_node = route[i]
        next_node = route[i+1]
        
        dist = dist_matrix[curr_node][next_node]
        total_distance += dist
        
        arrival_time = current_time + time_matrix[curr_node][next_node]
        wait_time = 0.0
        lateness = 0.0
        
        step_title = f"Step {i+1}:\n{curr_node} -> {next_node}"
        calc_str = f"• Distance = {dist:.3f} km\n• Arrival (A) = {arrival_time:.3f} h\n"

        bd = speed_breakdown.get(curr_node, {}).get(next_node, [])
        if bd:
            comp = "; ".join(f"{e['fclass']} @{e['speed_kmh']:.1f} km/h = {e['dist_km']:.2f} km" for e in bd)
            calc_str += f"• Road mix (fclass) = {comp}\n"

        if next_node != 'Depot':
            ready_time = float(time_windows[next_node][0])
            due_date = float(time_windows[next_node][1])
            
            if arrival_time < ready_time:
                wait_time = ready_time - arrival_time
                total_wait_penalty += (wait_time * 10)
                current_time = ready_time + 0.1
                calc_str += f"• Wait Time: node ready time is {ready_time:.2f} h. Vehicle idles for {wait_time:.3f} hours.\n"
                calc_str += f"• Wait Penalty = {wait_time:.3f} * 10 = {wait_time * 10:.2f}\n"
                calc_str += "• Lateness Penalty = 0.\n"
            else:
                if arrival_time > due_date:
                    lateness = arrival_time - due_date
                    total_lateness_penalty += (lateness * 50)
                    calc_str += "• Wait Time = 0.\n"
                    calc_str += f"• Lateness: node due date is {due_date:.2f} h. Late by {lateness:.3f} hours.\n"
                    calc_str += f"• Lateness Penalty (P) = {lateness:.3f} * 50 = {lateness * 50:.2f}\n"
                else:
                    calc_str += "• Wait Time = 0. Penalty: 0.\n• Lateness Penalty = 0.\n"
                current_time = arrival_time + 0.1
            calc_str += f"• Departure = {current_time:.3f} h"
        else:
            current_time = arrival_time
            calc_str += f"• Arrival at Depot = {current_time:.3f} h. No penalty."
            
        steps_log.append((step_title, calc_str))
            
    total_time = current_time - 8.00
    overtime_penalty = max(0, total_time - 8.0) * 100
    excess_dist_penalty = max(0, total_distance - MAX_DIST_KM) * 20
    
    Z = total_distance + total_lateness_penalty + total_wait_penalty + overtime_penalty + excess_dist_penalty
    
    concl_title = "CONCLUSION"
    concl_str = (f"Total Distance (Σ c_ij) = {total_distance:.2f} km. Max Dist Penalty = {excess_dist_penalty:.2f}\n"
                 f"Total Lateness Penalty = {total_lateness_penalty:.2f}\n"
                 f"Total Wait Penalty = {total_wait_penalty:.2f}\n"
                 f"Total Operational Time = {total_time:.3f} hours.\n"
                 f"Overtime Penalty = {overtime_penalty:.2f}\n\n"
                 f"Final Z Score = {total_distance:.2f} + {total_wait_penalty:.2f} + {total_lateness_penalty:.2f} + {overtime_penalty:.2f} + {excess_dist_penalty:.2f} = {Z:.2f}")
    steps_log.append((concl_title, concl_str))
    
    return Z, total_distance, total_lateness_penalty, total_wait_penalty, overtime_penalty, excess_dist_penalty, steps_log

def nearest_neighbor(depot, unvisited_list):
    route = [depot]
    current = depot
    unvisited = unvisited_list.copy()
    while unvisited:
        best_dist = float('inf')
        next_node = None
        for candidate in unvisited:
            dist = dist_matrix[current][candidate]
            if dist < best_dist:
                best_dist = dist
                next_node = candidate
        route.append(next_node)
        unvisited.remove(next_node)
        current = next_node
    route.append(depot)
    return route

customers = [n for n in nodes.keys() if n != 'Depot']
nn_route = nearest_neighbor('Depot', customers)
nn_z, nn_dist, nn_late, nn_wait, nn_ot, nn_ex, nn_steps = evaluate_route_detailed(nn_route)

def two_opt_search(initial_route):
    best_route = initial_route[:]
    best_cost = evaluate_route_detailed(best_route)[0]
    improved = True
    
    while improved:
        improved = False
        for i in range(1, len(best_route) - 2):
            for j in range(i + 1, len(best_route) - 1):
                new_route = best_route[:i] + best_route[i:j+1][::-1] + best_route[j+1:]
                new_cost = evaluate_route_detailed(new_route)[0]
                if new_cost < best_cost:
                    best_cost = new_cost
                    best_route = new_route
                    improved = True
    return best_route

opt_route = two_opt_search(nn_route)
opt_z, opt_dist, opt_late, opt_wait, opt_ot, opt_ex, opt_steps = evaluate_route_detailed(opt_route)


IndentationError: unexpected indent (2976080328.py, line 15)

## 4. Visualization

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from shapely import wkt
from shapely.geometry import Point, LineString, box
import json

print("1. Loading coordinate matrix...")
with open('matrix.json', 'r') as f:
    matrix_data = json.load(f)
nodes_epsg3857 = matrix_data['nodes_epsg3857']

oneway_nodes = ['N7', 'N8', 'N9', 'N10', 'N15']
df_roads = pd.read_csv('roads_shanghai.csv', sep='\t')

def plot_on_ax(ax, title, target_nodes_keys, margin, is_zoom=False):
    x_coords = [nodes_epsg3857[k][0] for k in target_nodes_keys]
    y_coords = [nodes_epsg3857[k][1] for k in target_nodes_keys]

    min_x, max_x = min(x_coords) - margin, max(x_coords) + margin
    min_y, max_y = min(y_coords) - margin, max(y_coords) + margin
    area_box = box(min_x, min_y, max_x, max_y)
    
    drawn_lines = 0
    for idx, row in df_roads.iterrows():
        try:
            geom = wkt.loads(row['geometry'])
        except:
            continue
        if not isinstance(geom, LineString):
            continue
            
        if geom.intersects(area_box):
            x, y = geom.xy
            fclass = row['fclass']
            color = 'gray'
            linewidth = 0.8
            if fclass in ['motorway', 'primary']:
                color = 'red'; linewidth = 2.0
            elif fclass in ['secondary', 'tertiary']:
                color = 'orange'; linewidth = 1.5
            elif fclass == 'residential':
                color = 'lightgray'; linewidth = 0.6
                
            ax.plot(x, y, color=color, linewidth=linewidth, alpha=0.7)
            drawn_lines += 1

    if 'Depot' in target_nodes_keys:
        ax.scatter(nodes_epsg3857['Depot'][0], nodes_epsg3857['Depot'][1], c='red', s=150, marker='s', label='Depot', zorder=5, edgecolors='black')

    for name in target_nodes_keys:
        if name != 'Depot':
            coords = nodes_epsg3857[name]
            if name in oneway_nodes:
                ax.scatter(coords[0], coords[1], c='magenta', s=200, marker='*', zorder=6, edgecolors='black')
            else:
                ax.scatter(coords[0], coords[1], c='blue', s=60, zorder=5, edgecolors='black')
            
            offset = 15 if is_zoom else 20
            ax.annotate(name, (coords[0]+offset, coords[1]+offset), fontsize=10 if is_zoom else 9, fontweight='bold', color='darkblue')

    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel("Web Mercator X", fontsize=12)
    ax.set_ylabel("Web Mercator Y", fontsize=12)
    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.grid(True, linestyle='--', alpha=0.5)

    if not is_zoom:
        ax.plot([],[], color='red', linewidth=2.0, label='Highway / Primary')
        ax.plot([],[], color='orange', linewidth=1.5, label='Urban Road (Secondary)')
        ax.plot([],[], color='lightgray', linewidth=0.6, label='Residential Lane')
        ax.scatter([],[], c='magenta', s=200, marker='*', edgecolors='black', label='Severely Affected by One-Way')
        ax.legend(loc='upper right')
        
    return drawn_lines

print("2. Generating combined map (subplots)...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))
all_keys = list(nodes_epsg3857.keys())
lines1 = plot_on_ax(ax1, "Full Map (Overall Delivery Area)", all_keys, 1000)

zoom_keys = ['N7', 'N8', 'N9', 'N10', 'N11', 'N12', 'N13', 'N14', 'N15']
lines2 = plot_on_ax(ax2, "Zoom-In Map (Critical One-Way Zone)", zoom_keys, 200, is_zoom=True)

plt.tight_layout()
plt.show()
